# Opbygning af et genanvendeligt aktuarielt prisfastsættelsesbibliotek med PROC FCMP

## Sammenfatning

Et skadesforsikringsselskab (ejendom og ansvar) indkapsler sin centrale prisfastsættelsesmatematik — nettopræmie, omkostnings-/risikotillæg, begrænset-fluktuations-kredibilitetsblanding og tilbagediskonteret hensættelsesestimering — som brugerdefinerede funktioner og en subrutine med flere returværdier i **PROC FCMP**. Det kompilerede bibliotek registreres via systemoptionen `CMPLIB=` og kaldes derefter række for række fra et DATA-step, der prisfastsætter en syntetisk portefølje på 100 policer. Hvert prisfastsat tal i denne notebook — kredibilitetsfaktoren `z`, den blandede nettopræmie, den opkrævede præmie og den tilbagediskonterede skadeshensættelse — beregnes af disse kompilerede rutiner, ikke ved indlejret aritmetik. Resultatet: den implicitte skadeprocent lander på 60.5% (Land), 55.8% (Forstad) og 63.6% (By) — behageligt under 100% i alle segmenter, hvilket bekræfter, at den tillagte præmie dækker de forventede skader, mens prisfastsættelsestrinnet forbliver rent og reviderbart.

## Datakilder

| Datasæt | Rækker | Beskrivelse | Nøglevariable |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Syntetisk portefølje af skadesforsikringspolicer genereret direkte med `rand()` | `policy_id`, `region` (By/Forstad/Land), `years_insured`, `exposure` (bilår), `claim_count` (Poisson), `incurred_loss` (gamma-skadesbeløb x antal), `class_pure_prem` (manuel takst efter region) |

DATA-steppet løber over et bredere `policy_id`-interval, men dette miljø kører uden licens, så output er begrænset til de første **100 observationer** — den prisfastsatte bog nedenfor afspejler netop disse 100 policer.

# Opbygning af et genanvendeligt aktuarielt prisfastsættelsesbibliotek med PROC FCMP

Aktuarer gentager de samme beregninger på tværs af prisfastsættelse, hensættelse og rapportering: omregn skader til en *nettopræmie*, anvend *omkostnings- og risikotillæg* for at nå frem til en opkrævet præmie, bland en enkelt polices egen erfaring med en klassetakst ved hjælp af *kredibilitet*, og *tilbagediskonter* fremtidige pengestrømme til nutidsværdi. At skrive disse formler igen i hvert DATA-step lægger op til copy-paste-fejl og gør ændringer af antagelser smertefulde.

**PROC FCMP** (SAS'/Jenners funktionskompilator) lader os definere hver formel én gang som en navngiven funktion eller subrutine, gemme de kompilerede rutiner i et bibliotek og derefter kalde dem som enhver indbygget funktion. I denne notebook:

1. Kompilerer vi et lille aktuarielt funktionsbibliotek med `PROC FCMP`.
2. Registrerer vi det for sessionen med systemoptionen `CMPLIB=`.
3. Genererer vi en syntetisk skadesforsikringsportefølje.
4. Prisfastsætter vi hver police ved at kalde vores brugerdefinerede funktioner og en subrutine med flere returværdier fra et enkelt DATA-step.
5. Sammenfatter og fortolker vi den prisfastsatte portefølje.

## Trin 1 — Generér en syntetisk policeportefølje

Vi simulerer en bog af igangværende bilpolicer (de første 100 prisfastsættes nedenfor under loft for miljø uden licens). Hver police tilhører en takstregion med sin egen manuelle *nettopræmie* (forventet skade pr. bilår). Antal skader følger en Poisson-proces skaleret med eksponering, og skadesbeløb følger en gammafordeling, så `incurred_loss` er et realistisk sammensat (Poisson x gamma) skadesbeløb. `years_insured` driver senere kredibilitetsvægten. Et fast startpunkt via `call streaminit` gør data reproducerbare.

In [1]:
data portfolio;
    CALL streaminit(20260531);
    LÆNGDE region $9;
    GØR policy_id = 10001 TIL 12000;
        /* Tildel en takstregion */
        u = rand('uniform');
        HVIS u < 0.40 SÅ GØR; region = 'By';       base_pp = 820; lambda = 0.18; SLUT;
        ELLERS HVIS u < 0.75 SÅ GØR; region = 'Forstad'; base_pp = 540; lambda = 0.11; SLUT;
        ELLERS GØR; region = 'Land';     base_pp = 360; lambda = 0.07; SLUT;

        /* Anciennitet (forsikringsår) og eksponering (bilår) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Sammensat skadesproces: Poisson-frekvens, gamma-skadesbeløb */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        GØR c = 1 TIL claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        SLUT;
        incurred_loss = round(incurred_loss, 0.01);

        /* Manuel klassenettopræmie for denne polices eksponering */
        class_pure_prem = round(base_pp * exposure, 0.01);

        UDDATA;
    SLUT;
    BEHOLD policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
KØR;

PROCEDURE UDSKRIV data=portfolio(obs=8) noobs MÆRKAT;
    MÆRKAT policy_id="Policenr." region="Region" years_insured="Forsikringsår"
          exposure="Eksponering (bilår)" claim_count="Antal skader"
          incurred_loss="Indtrufne skader" class_pure_prem="Klassens nettopræmie";
    TITEL "Første 8 simulerede policer";
KØR;


                                              Første 8 simulerede policer                                               

Policenr.   Region   Forsikringsår   Eksponering (bilår)  Antal skader  Indtrufne skader   Klassens nettopræmie
    10001  Land                  5                  1.36             0                 0                  489.6
    10002  By                    8                  0.97             1           3432.28                  795.4
    10003  By                    2                  1.53             2           7155.92                 1254.6
    10004  Forstad               9                   2.4             0                 0                   1296
    10005  Land                  5                  0.99             0                 0                  356.4
    10006  Forstad               1                  0.83             0                 0                  448.2
    10007  Land                  5                  0.97             0                 0      


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.43 seconds
  cpu   0.43 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Trin 2 — Kompilér det aktuarielle funktionsbibliotek

Nu selve kernen i notebooken. `PROC FCMP` med `OUTLIB=work.actfuncs.pricing` kompilerer fire funktioner og én subrutine til pakken `pricing` i datasættet `work.actfuncs`:

- **`pure_premium`** — observeret skade pr. enhed eksponering (frekvens x skadesbeløb kombineret).
- **`credibility_z`** — begrænset-fluktuations-kredibilitetsfaktor `Z = sqrt(n / (n + k))`, hvor `n` er policens eksponeringsår, og `k` er en tuning-konstant.
- **`charged_premium`** — anvender et proportionalt risikotillæg og en fast omkostningsprocent på en skadesomkostning for at nå frem til den præmie, vi rent faktisk opkræver.
- **`pv_reserve`** — nutidsværdi af en fremtidig skadesudbetaling, `FV / (1+r)**t`, brugt til at tilbagediskontere hensættelser.
- **`blend_premium`** (en SUBROUTINE) — bruger `OUTARGS` til at returnere *to* værdier på én gang: den kredibilitetsvægtede nettopræmie og den anvendte kredibilitetsfaktor, så DATA-steppet får begge dele i ét kald.

Hver rutine afsluttes med `ENDSUB`, og subrutinen navngiver sine skrivbare argumenter med `OUTARGS`.

In [2]:
PROCEDURE fcmp outlib=work.actfuncs.pricing;

    /* Observeret nettopræmie: skadesomkostning pr. enhed eksponering */
    function pure_premium(loss, exposure);
        HVIS exposure <= 0 SÅ RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Begrænset-fluktuations-kredibilitet: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        HVIS n_years <= 0 SÅ RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Opkrævet præmie = skadesomkostning tillagt risiko- og omkostningstillæg */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        HVIS expense_ratio >= 1 SÅ RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Nutidsværdi af en fremtidig skadesudbetaling tilbagediskonteret t år til rente r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Kredibilitetsblanding: returnerer den vægtede nettopræmie OG den anvendte Z */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

KØR;



               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Trin 3 — Registrér biblioteket med CMPLIB=

At kompilere rutinerne er ikke nok; SAS/Jenner skal fortælles, hvor de skal findes, når et senere DATA-step eller en procedure refererer til et navn, den ikke genkender som indbygget. `CMPLIB=` peger på datasættet (ikke det trecifrede pakkenavn), der indeholder den kompilerede kode. Efter denne `OPTIONS`-sætning kan hver funktion og subrutine ovenfor kaldes ved navn.

In [3]:
INDSTILLINGER cmplib=work.actfuncs;



NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Trin 4 — Prisfastsæt porteføljen med de brugerdefinerede funktioner

Prisfastsættelses-DATA-steppet er nu næsten fri for aritmetik — den aktuarielle hensigt fremgår direkte af funktionsnavnene. For hver police:

1. Beregner vi policens egen observerede nettopræmie med `pure_premium`.
2. Kalder vi subrutinen `blend_premium` for at kredibilitetsvægte den mod den regionale klassetakst, og henter både den blandede skadesomkostning og kredibilitetsfaktoren `z` via `OUTARGS`.
3. Opskriver vi den blandede skadesomkostning til en opkrævet præmie med et risikotillæg på 12 % og en omkostningsprocent på 25 % via `charged_premium`.
4. Estimerer vi den fortsat åbne skadeshensættelse som 35 % af policens indtrufne skade og tilbagediskonterer den tre år ved 4 % til nutidsværdi med `pv_reserve`.

Bemærk, at subrutinens returargumenter (`blended_pp`, `z`) skal initialiseres, før `CALL` kaldes. Hensættelsens nutidsværdi varierer fra police til police, fordi den drives af hver polices faktiske indtrufne skade — skadefri policer bærer en hensættelse på nul, så deres `reserve_pv` også er nul.

In [4]:
data rated;
    SÆT portfolio;

    /* 1. Policens egen skadeserfaring som nettopræmie */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Kredibilitetsvægt egen erfaring mod klassens sats.
          k = 4 eksponeringsår for næsten fuld kredibilitet. */
    blended_pp = .;
    z = .;
    CALL blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Konverter blandet skadesomkostning (pr. bilår) til opkrævet præmie */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Udestående skadeshensættelse = 35% af indtrufne skader, forventes
          at afvikles om 3 år; tilbagediskonter til nutidsværdi ved 4%. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    BEHOLD policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
KØR;

PROCEDURE UDSKRIV data=rated(obs=10) noobs MÆRKAT;
    MÆRKAT policy_id="Policenr." region="Region" years_insured="Forsikringsår"
          exposure="Eksponering (bilår)" own_pp="Egen nettopræmie"
          blended_pp="Blandet nettopræmie" z="Kredibilitet (z)"
          premium="Præmie" reserve_pv="Hensættelse (NV)";
    TITEL "Prisfastsat portefølje (første 10 policer)";
    VARIABEL policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
KØR;


                                       Prisfastsat portefølje (første 10 policer)                                       

Policenr.   Region   Forsikringsår   Eksponering (bilår)   Egen nettopræmie   Blandet nettopræmie  Kredibilitet (z)   Præmie   Hensættelse (NV)
    10001  Land                  5                  1.36                  0                 91.67             0.745   186.18                  0
    10002  By                    8                  0.97            3538.43               3039.59             0.816  4402.95            1067.95
    10003  By                    2                  1.53            4677.07               3046.88             0.577  6961.51            2226.55
    10004  Forstad               9                   2.4                  0                 90.69             0.832   325.03                  0
    10005  Land                  5                  0.99                  0                 91.67             0.745   135.52                  0
    10006  For


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Trin 5 — Sammenfat den prisfastsatte portefølje

Med hver police prisfastsat gennem det samme kompilerede bibliotek kan vi rulle porteføljen op efter region. Vi rapporterer gennemsnitlig opkrævet præmie, gennemsnitlig kredibilitetsfaktor, samlede indtrufne skader og den samlede tilbagediskonterede skadeshensættelse, og udleder derefter den implicitte skadeprocent pr. segment, så vi kan se, om den tillagte præmie dækker de forventede skader.

In [5]:
PROCEDURE GENNEMSNIT data=rated mean sum maxdec=2 nonobs;
    KLASSE region;
    VARIABEL premium z incurred_loss reserve_pv;
    MÆRKAT region="Region" premium="Præmie" z="Kredibilitet (z)"
          incurred_loss="Indtrufne skader" reserve_pv="Hensættelse (NV)";
    TITEL "Porteføljesammendrag efter region";
KØR;

/* Implicit skadeprocent = indtrufne skader / opkrævet præmie, plus den
   tilbagediskonterede hensættelse segmentet bærer, efter region. */
PROCEDURE SQL;
    TITEL "Implicit skadeprocent og hensættelse (NV) efter region";
    VÆLG region,
           count(*)                              AS antal_policer,
           sum(incurred_loss)                    AS samlede_skader     format=dollar12.2,
           sum(premium)                          AS samlet_praemie     format=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS skadeprocent       format=percent8.1,
           sum(reserve_pv)                       AS samlet_nv_hensaettelse format=dollar12.2
    FROM rated
    GROUP EFTER region
    ORDER EFTER region;
QUIT;
TITEL;


                                           Porteføljesammendrag efter region                                            

                                                  The MEANS Procedure

                                          Analysis Variable : premium Præmie

        Region             Mean            Sum
        --------------------------------------
        By              1987.17       67563.93
        Forstad          813.04       34147.74
        Land             476.61       11438.62
        --------------------------------------

                                         Analysis Variable : z Kredibilitet (z)

        Region             Mean            Sum
        --------------------------------------
        By                 0.70          23.90
        Forstad            0.68          28.36
        Land               0.71          17.14
        --------------------------------------

                                   Analysis Variable : incurred_loss Indtrufne ska


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Fortolkning af resultaterne

Prisfastsættelses-DATA-steppet staver aldrig en eneste diskonterings- eller kredibilitetsformel ud direkte — det kalder bare `pure_premium`, `blend_premium`, `charged_premium` og `pv_reserve`. Det er gevinsten ved **PROC FCMP**: de aktuarielle antagelser bor i ét kompileret bibliotek, der kan enhedstestes, versionsstyres og genbruges på tværs af prisfastsættelse, hensættelse og rapporteringsopgaver. At ændre kredibilitetskonstanten `k`, risikotillægget eller omkostningsprocenten er én enkelt rettelse i biblioteket, ikke en gennemgang af hvert eneste program.

Læsning af den prisfastsatte bog og den regionale opsummering:

- **Kredibilitet (`z`)** stiger med `years_insured`, præcis som den begrænset-fluktuations-formel `Z = sqrt(n / (n + k))` dikterer. Blandt de første ti policer opnår den etårige Forstad-police (10006) `z = 0.447`, den toårige By-police (10003) `z = 0.577`, mens den niårige Forstad-police (10004) når `z = 0.832`. Policer med tynd erfaring trækkes mod den regionale klassetakst; langvarige policer læner sig op ad egne skader.
- **Blanding i praksis:** skadefri policer (størstedelen af bogen) har `own_pp = 0`, så `blend_premium` returnerer en `blended_pp` tæt på `(1 - z)` gange klassetaksten — f.eks. lander police 10001 (Land, `z = 0.745`) på `91.67` mod en klassetakst på `360`/bilår. De to By-policer, der faktisk pådrog sig skader, 10002 og 10003, trækker i stedet deres blandede skadesomkostning op mod deres egen høje erfaring (`3039.59` og `3046.88`).
- **Opkrævet præmie** ligger over den blandede skadesomkostning, fordi `charged_premium` lægger et risikotillæg på 12 % til og opskriver for en omkostningsprocent på 25 %, en fast multiplikator på `1.12 / 0.75 = 1.493` på skadesomkostningen. Gennemsnitlig opkrævet præmie ligger på `476.61` (Land), `813.04` (Forstad) og `1,987.17` (By).
- **Tilbagediskonterede hensættelser:** `pv_reserve` tilbagediskonterer hver polices udestående skadeshensættelse (35 % af indtrufne skader) tre år ved 4 %, en nutidsværdifaktor på `0.889`. Skadefri policer bærer `reserve_pv = 0`; de to By-skadesager bidrager med `1,067.95` og `2,226.55`. Samlet set holder bogen en tilbagediskonteret hensættelse på `$2,151.56` (Land), `$5,932.52` (Forstad) og `$13,364.13` (By).
- **Implicitte skadeprocenter** lander på 60.5% (Land), 55.8% (Forstad) og 63.6% (By) — alle behageligt under 100 %, så den tillagte præmie dækker de forventede skader i alle segmenter. By-segmentet er det hedeste, drevet af dets to store simulerede skader; en reel taksteftersyn ville teste, om dette signal holder over flere skadeår, før den manuelle takst justeres.

Subrutinen `blend_premium` demonstrerer også `OUTARGS`-idiomet til at returnere flere resultater fra ét `CALL` — den blandede skadesomkostning og kredibilitetsfaktoren `z` kommer tilbage sammen — hvilket undgår separate funktionskald og holder den police-vise prisfastsættelseslogik kompakt og reviderbar.